# Aggregate Classification Sweep Results

Load classification sweep results for multiple campaigns, select one top percentage, and average the results across campaigns.

In [17]:
from pathlib import Path

import pandas as pd

## Parameters

In [18]:
Campaigns = ["Iran_1", "UAE", "Venezuela"] # "China_1", "Cuba", "Russia_1"
top_percentage = 100

In [19]:
def find_project_root(start_path):
    """Find the project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the project root containing AGENTS.md.")


def get_results_path(project_root, campaign):
    """Build the classification sweep results path for a campaign.

    Args:
        project_root: Root directory of the project.
        campaign: Campaign directory name.

    Returns:
        Path to the campaign's classification sweep CSV file.
    """
    return (
        project_root
        / "results"
        / "Labeled_Datasets"
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / "classification_sweep"
        / "classification_sweep_results.csv"
    )

In [20]:
if not Campaigns:
    raise ValueError("Campaigns must contain at least one campaign name.")

project_root = find_project_root(Path.cwd())
campaign_results = {}

for campaign in Campaigns:
    results_path = get_results_path(project_root, campaign)
    campaign_df = pd.read_csv(results_path)
    campaign_df = campaign_df.loc[
        campaign_df["top_percentage"].eq(top_percentage)
    ]
    campaign_df["indicators_configuration"] = campaign_df[
        "indicators_configuration"
    ].str.replace(
        " (Unweighted Clustering, No Square Clustering)",
        "",
        regex=False,
    )
    campaign_results[campaign] = campaign_df

configuration_columns = [
    "topic_col",
    "account_relations_configuration",
    "indicators_configuration",
]
configuration_presence_records = []
common_configuration_values = {}

for configuration_column in configuration_columns:
    values_by_campaign = {
        campaign: set(campaign_df[configuration_column].dropna().unique())
        for campaign, campaign_df in campaign_results.items()
    }
    common_configuration_values[configuration_column] = set.intersection(
        *values_by_campaign.values()
    )
    all_configuration_values = set.union(*values_by_campaign.values())

    for configuration_value in sorted(all_configuration_values):
        present_in = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value in values
        ]
        missing_from = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value not in values
        ]
        configuration_presence_records.append(
            {
                "configuration": configuration_column,
                "value": configuration_value,
                "present_in": ", ".join(present_in),
                "missing_from": ", ".join(missing_from) or "None",
                "present_in_all_datasets": not missing_from,
            }
        )

configuration_presence_df = pd.DataFrame(configuration_presence_records)
print("Configuration availability by campaign:")
display(configuration_presence_df)

requested_summary_metric_columns = {
    "all_authors_auc": "ROC_AUC",
    "all_authors_auc_pr": "PR_AUC",
}
metric_presence_records = []
summary_metric_columns = {}
for metric_column, metric_name in requested_summary_metric_columns.items():
    present_in = [
        campaign
        for campaign, campaign_df in campaign_results.items()
        if metric_column in campaign_df
        and campaign_df[metric_column].notna().any()
    ]
    missing_from = [
        campaign for campaign in Campaigns if campaign not in present_in
    ]
    metric_presence_records.append(
        {
            "metric": metric_name,
            "source_column": metric_column,
            "present_in": ", ".join(present_in),
            "missing_from": ", ".join(missing_from) or "None",
            "present_in_all_datasets": not missing_from,
        }
    )
    if not missing_from:
        summary_metric_columns[metric_column] = metric_name

metric_presence_df = pd.DataFrame(metric_presence_records)
print("Metric availability by campaign:")
display(metric_presence_df)

all_results_df = pd.concat(campaign_results.values(), ignore_index=True)
for configuration_column, common_values in common_configuration_values.items():
    all_results_df = all_results_df.loc[
        all_results_df[configuration_column].isin(common_values)
    ]
all_results_df = all_results_df.reset_index(drop=True)
group_columns = all_results_df.select_dtypes(exclude="number").columns.tolist()
aggregated_df = (
    all_results_df.groupby(group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(group_columns)
    .reset_index(drop=True)
)

display(aggregated_df)

classification_group_columns = configuration_columns
classification_metric_columns = [
    column
    for column in all_results_df.columns
    if column.startswith(("graph_authors_", "all_authors_"))
    and all(
        column in campaign_df and campaign_df[column].notna().any()
        for campaign_df in campaign_results.values()
    )
]
classification_metrics_df = (
    all_results_df[
        classification_group_columns + classification_metric_columns
    ]
    .groupby(classification_group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(classification_group_columns)
    .reset_index(drop=True)
)

display(classification_metrics_df)

topic_col_aggregation_df = (
    all_results_df.groupby("topic_col", as_index=False, dropna=False)[
        list(summary_metric_columns)
    ]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("topic_col")
    .reset_index(drop=True)
)
display(topic_col_aggregation_df)

account_relations_aggregation_df = (
    all_results_df.groupby(
        "account_relations_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns)]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("account_relations_configuration")
    .reset_index(drop=True)
)
display(account_relations_aggregation_df)

indicators_aggregation_df = (
    all_results_df.groupby(
        "indicators_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns) + ["indicators_runtime_seconds"]]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("indicators_configuration")
    .reset_index(drop=True)
)
display(indicators_aggregation_df)

Configuration availability by campaign:


,configuration,value,present_in,missing_from,present_in_all_datasets
0,topic_col,BERTopic_topic_10,"Iran_1, UAE, Venezuela",None,True
1,topic_col,BERTopic_topic_128,"Iran_1, UAE, Venezuela",None,True
2,topic_col,BERTopic_topic_256,"Iran_1, UAE, Venezuela",None,True
3,topic_col,BERTopic_topic_512,"Iran_1, UAE, Venezuela",None,True
4,topic_col,K_Means_topic_128,Iran_1,"UAE, Venezuela",False
5,topic_col,K_Means_topic_16,UAE,"Iran_1, Venezuela",False
6,topic_col,K_Means_topic_60,Venezuela,"Iran_1, UAE",False
7,account_relations_configuration,interactions,"Iran_1, UAE, Venezuela",None,True
8,account_relations_configuration,similarity,"Iran_1, UAE, Venezuela",None,True
9,account_relations_configuration,similarity+interactions,"Iran_1, UAE, Venezuela",None,True


Metric availability by campaign:


,metric,source_column,present_in,missing_from,present_in_all_datasets
0,ROC_AUC,all_authors_auc,"Iran_1, UAE, Venezuela",None,True
1,PR_AUC,all_authors_auc_pr,"Iran_1, UAE, Venezuela",None,True


,topic_col,account_relations_configuration,indicators_configuration,model,top_percentage,indicators_runtime_seconds,n_graph_authors,n_all_authors,graph_authors_auc,graph_authors_f1,graph_authors_auc_pr,graph_authors_recall,graph_authors_precision,all_authors_auc,all_authors_f1,all_authors_auc_pr,all_authors_recall,all_authors_precision
0,BERTopic_topic_10,interactions,All,logistic_regression,100.0,50.207526,10077.666667,10077.666667,0.898117,0.712021,0.781755,0.591521,0.903547,0.898117,0.712021,0.781755,0.591521,0.903547
1,BERTopic_topic_10,interactions,All,random_forest,100.0,50.207526,10077.666667,10077.666667,0.952737,0.872080,0.910051,0.794897,0.970860,0.952737,0.872080,0.910051,0.794897,0.970860
2,BERTopic_topic_10,interactions,All,xgboost,100.0,50.207526,10077.666667,10077.666667,0.957441,0.888158,0.914599,0.826851,0.962329,0.957441,0.888158,0.914599,0.826851,0.962329
3,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,logistic_regression,100.0,50.207526,10077.666667,10077.666667,0.897433,0.716294,0.779653,0.595505,0.903654,0.897433,0.716294,0.779653,0.595505,0.903654
4,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,random_forest,100.0,50.207526,10077.666667,10077.666667,0.951026,0.875151,0.908795,0.800168,0.970246,0.951026,0.875151,0.908795,0.800168,0.970246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,BERTopic_topic_512,text_similarity+interactions,All but Betweenness and Harmonic,random_forest,100.0,4410.398485,10077.666667,10077.666667,0.992862,0.972284,0.984804,0.953649,0.991799,0.992862,0.972284,0.984804,0.953649,0.991799
248,BERTopic_topic_512,text_similarity+interactions,All but Betweenness and Harmonic,xgboost,100.0,4410.398485,10077.666667,10077.666667,0.993504,0.974178,0.985488,0.958985,0.989930,0.993504,0.974178,0.985488,0.958985,0.989930
249,BERTopic_topic_512,text_similarity+interactions,Core Number Only,logistic_regression,100.0,4410.398485,10077.666667,10077.666667,0.873270,0.529928,0.597384,0.501888,0.561290,0.873270,0.529928,0.597384,0.501888,0.561290
250,BERTopic_topic_512,text_similarity+interactions,Core Number Only,random_forest,100.0,4410.398485,10077.666667,10077.666667,0.948734,0.718196,0.779188,0.694506,0.866908,0.948734,0.718196,0.779188,0.694506,0.866908


,topic_col,account_relations_configuration,indicators_configuration,graph_authors_auc,graph_authors_f1,graph_authors_auc_pr,graph_authors_recall,graph_authors_precision,all_authors_auc,all_authors_f1,all_authors_auc_pr,all_authors_recall,all_authors_precision
0,BERTopic_topic_10,interactions,All,0.936098,0.824086,0.868802,0.737756,0.945579,0.936098,0.824086,0.868802,0.737756,0.945579
1,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,0.934495,0.826170,0.866893,0.740419,0.944994,0.934495,0.826170,0.866893,0.740419,0.944994
2,BERTopic_topic_10,interactions,Core Number Only,0.773825,0.360028,0.448062,0.276603,0.534875,0.773825,0.360028,0.448062,0.276603,0.534875
3,BERTopic_topic_10,similarity,All,0.979419,0.912130,0.953277,0.864945,0.971856,0.979419,0.912130,0.953277,0.864945,0.971856
4,BERTopic_topic_10,similarity,All but Betweenness and Harmonic,0.978537,0.907629,0.947953,0.859013,0.970477,0.978537,0.907629,0.947953,0.859013,0.970477
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,BERTopic_topic_512,text_similarity,All but Betweenness and Harmonic,0.980769,0.916305,0.942945,0.884126,0.959172,0.980769,0.916305,0.942945,0.884126,0.959172
80,BERTopic_topic_512,text_similarity,Core Number Only,0.924167,0.646604,0.709474,0.624437,0.744318,0.924167,0.646604,0.709474,0.624437,0.744318
81,BERTopic_topic_512,text_similarity+interactions,All,0.976930,0.915754,0.947299,0.889053,0.954615,0.976930,0.915754,0.947299,0.889053,0.954615
82,BERTopic_topic_512,text_similarity+interactions,All but Betweenness and Harmonic,0.980261,0.915944,0.941579,0.881907,0.963065,0.980261,0.915944,0.941579,0.881907,0.963065


,topic_col,ROC_AUC,PR_AUC
0,BERTopic_topic_10,0.945314,0.859693
1,BERTopic_topic_128,0.945265,0.852223
2,BERTopic_topic_256,0.943722,0.839869
3,BERTopic_topic_512,0.947240,0.848214


,account_relations_configuration,ROC_AUC,PR_AUC
0,interactions,0.881200,0.722417
1,similarity,0.952883,0.878032
2,similarity+interactions,0.952451,0.877350
3,similarity+text_similarity,0.957761,0.866927
4,similarity+text_similarity+interactions,0.957561,0.867663
5,text_similarity,0.957703,0.868312
6,text_similarity+interactions,0.958137,0.869297


,indicators_configuration,ROC_AUC,PR_AUC,indicators_runtime_seconds
0,All,0.968838,0.930288,2536.902855
1,All but Betweenness and Harmonic,0.972310,0.931875,2536.902855
2,Core Number Only,0.895009,0.687835,2536.902855


In [21]:
campaigns_name = "_".join(Campaigns).lower().replace(" ", "_")
percentage_name = f"{top_percentage:g}"
output_folder_name = f"top_percentage_{percentage_name}_{campaigns_name}"
output_dir = (
    project_root
    / "results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated"
    / output_folder_name
)
output_dir.mkdir(parents=True, exist_ok=True)

output_stem = "aggregated_classification_sweep_results"
csv_path = output_dir / f"{output_stem}.csv"
latex_path = output_dir / f"{output_stem}.tex"
metrics_output_stem = "aggregated_classification_metrics"
metrics_csv_path = output_dir / f"{metrics_output_stem}.csv"
metrics_latex_path = output_dir / f"{metrics_output_stem}.tex"

aggregated_df.to_csv(csv_path, index=False)
aggregated_df.to_latex(latex_path, index=False, float_format="%.3f")
classification_metrics_df.to_csv(metrics_csv_path, index=False)
classification_metrics_df.to_latex(
    metrics_latex_path, index=False, float_format="%.3f"
)

summary_aggregations = {
    "topic_col": topic_col_aggregation_df,
    "account_relations_configuration": account_relations_aggregation_df,
    "indicators_configuration": indicators_aggregation_df,
}
for aggregation_name, summary_df in summary_aggregations.items():
    summary_output_stem = f"{output_stem}_by_{aggregation_name}"
    summary_csv_path = output_dir / f"{summary_output_stem}.csv"
    summary_latex_path = output_dir / f"{summary_output_stem}.tex"
    summary_df.to_csv(summary_csv_path, index=False)
    summary_df.to_latex(
        summary_latex_path, index=False, float_format="%.3f"
    )
    print(f"Saved summary CSV: {summary_csv_path}")
    print(f"Saved summary LaTeX: {summary_latex_path}")

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX: {latex_path}")
print(f"Saved classification metrics CSV: {metrics_csv_path}")
print(f"Saved classification metrics LaTeX: {metrics_latex_path}")

Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/top_percentage_100_iran_1_uae_venezuela/aggregated_classification_sweep_results_by_topic_col.csv
Saved summary LaTeX: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/top_percentage_100_iran_1_uae_venezuela/aggregated_classification_sweep_results_by_topic_col.tex
Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/top_percentage_100_iran_1_uae_venezuela/aggregated_classification_sweep_results_by_account_relations_configuration.csv
Saved summary LaTeX: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/top_percentage_100_iran_1_uae_venezuela/aggregated_classification_sweep_results_by_account_relations_configuration.tex
Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_in